In [1]:
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def herb_to_doc(r):
    return {
        "text": f"""
        Herb: {r.get("name")}
        Helps with: {", ".join(r.get("commonly_used_for", []))}
        Properties: {", ".join(r.get("primary_properties", []))}
        Benefits: {", ".join(r.get("health_benefits", []))}
        Compounds: {", ".join(r.get("key_compounds", []))}
        """.lower().strip(),
        "type": "herb",
        "entity": r.get("name")
    }


def nutrient_to_doc(r):
    return {
        "text": f"""
        Nutrient: {r.get("entity")}
        Found in: {", ".join(r.get("food_sources", []))}
        Supports: {", ".join(r.get("target_systems", []))}
        Deficiency causes: {", ".join(r.get("deficiency_effects", []))}
        Interacts with: {", ".join(r.get("interacts_with", []))}
        """.lower().strip(),
        "type": "nutrient",
        "entity": r.get("entity")
    }


def disease_to_doc(r):
    return {
        "text": f"""
        Disease: {r.get("name")}
        Symptoms: {", ".join(r.get("symptoms", []))}
        Causes: {", ".join(r.get("causes", []))}
        """.lower().strip(),
        "type": "disease",
        "entity": r.get("name")
    }


def build_documents():
    herbs = load_json("../datasets/rag_json_docs/herbs.json")
    nutrients = load_json("../datasets/rag_json_docs/nutrients.json")
    diseases = load_json("../datasets/rag_json_docs/diseases.json")

    docs = []
    docs += [herb_to_doc(x) for x in herbs]
    docs += [nutrient_to_doc(x) for x in nutrients]
    docs += [disease_to_doc(x) for x in diseases]

    return docs

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)


def embed_docs(docs):
    texts = [d["text"] for d in docs]
    return embedding_model.embed_documents(texts)

c:\Users\Rakesh Kumar\VSCode\Medical_Assistant\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1784.51it/s]


In [4]:
from sklearn.cluster import KMeans


def cluster_by_type(docs, vectors, k=3):
    grouped = {}

    for i, doc in enumerate(docs):
        grouped.setdefault(doc["type"], []).append((i, doc))

    clustered = {}

    for dtype, items in grouped.items():
        idxs = [i for i, _ in items]
        vecs = [vectors[i] for i in idxs]

        kmeans = KMeans(n_clusters=k, random_state=42)
        labels = kmeans.fit_predict(vecs)

        clusters = {}
        for i, label in enumerate(labels):
            clusters.setdefault(label, []).append(idxs[i])

        clustered[dtype] = clusters

    return clustered

In [ ]:
from google import genai
import os
from dotenv import load_dotenv

client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))
MODEL = "gemini-2.5-flash"


def summarize_cluster(texts):
    combined = "\n".join(texts)

    prompt = f"""
Summarize the following into a compact medical knowledge paragraph:

{combined}
"""

    res = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return res.text.strip()

In [5]:
def build_raptor_docs(docs, vectors):
    clustered = cluster_by_type(docs, vectors)

    summaries = []

    for dtype, clusters in clustered.items():
        for cluster_id, indices in clusters.items():

            cluster_texts = [docs[i]["text"] for i in indices]

            summary = summarize_cluster(cluster_texts)

            summaries.append({
                "text": summary.lower().strip(),
                "type": f"{dtype}_cluster",
                "entity": None
            })

    return docs + summaries

In [6]:
from langchain_community.vectorstores import FAISS
from langchain_classic.schema import Document


def build_and_save():
    docs = build_documents()
    vectors = embed_docs(docs)

    all_docs = build_raptor_docs(docs, vectors)

    documents = [
        Document(
            page_content=d["text"],
            metadata={
                "type": d["type"],
                "entity": d["entity"]
            }
        )
        for d in all_docs
    ]

    db = FAISS.from_documents(documents, embedding_model)

    db.save_local("raptor_index_v1")

    print("✅ RAPTOR index saved")

In [10]:
build_and_save()

✅ RAPTOR index saved
